# 🎬 마케팅 영상 광고 효과 예측 에이전트

> **프로모션 영상을 업로드하면 Claude AI가 광고 효과를 종합 분석하여 예측 점수와 개선 방안을 제공합니다.**

---
### 📋 분석 항목
| 항목 | 설명 |
|------|------|
| 🎯 시각적 임팩트 | 첫 3초 후킹력, 색감, 구도 |
| 💬 메시지 명확성 | 핵심 메시지 전달력, CTA 명확도 |
| ❤️ 감성 공명도 | 감정 유발, 브랜드 친밀도 |
| 🔔 CTA 효과성 | 행동 유도 문구 명확도 |
| 🏷️ 브랜드 일관성 | 브랜드 톤앤매너 일치도 |
| 📊 전환 가능성 | 구매 의도 유발, ROI 예측 |

### ⚙️ 실행 순서
1. **Cell 1** — 패키지 설치
2. **Cell 2** — Anthropic API 키 입력
3. **Cell 3** — 분석 엔진 로딩
4. **Cell 4** — Claude 에이전트 정의
5. **Cell 5** — 리포트 렌더러 로딩
6. **Cell 6** — ▶ 영상 업로드 & 분석 실행
7. **Cell 7** *(선택)* — YouTube URL로 분석
8. **Cell 8** *(선택)* — 여러 영상 배치 비교 (A/B 테스트)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# [Cell 1] 패키지 설치
print('📦 필요한 패키지를 설치합니다...')
import subprocess, sys
pkgs = ['anthropic','opencv-python-headless','Pillow','numpy','ipywidgets']
for p in pkgs:
    subprocess.check_call([sys.executable,'-m','pip','install',p,'-q'])
print('✅ 패키지 설치 완료!')


In [ ]:
# [Cell 2] Anthropic API 키 설정
import os
from getpass import getpass
print('🔑 Anthropic API 키를 입력하세요')
print('   발급: https://console.anthropic.com')
ANTHROPIC_API_KEY = getpass('API Key: ')
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
if ANTHROPIC_API_KEY.startswith('sk-ant-'):
    print('✅ API 키가 설정되었습니다!')
else:
    print('⚠️  API 키 형식을 확인하세요 (sk-ant- 로 시작해야 합니다)')


In [ ]:
# [Cell 3] 분석 엔진
import anthropic, cv2, base64, json, time, numpy as np
from PIL import Image
from io import BytesIO
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])

def extract_keyframes(video_path, n_frames=8):
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30
    indices = np.linspace(0, total-1, n_frames, dtype=int)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret: continue
        img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        img.thumbnail((1280,720))
        buf = BytesIO()
        img.save(buf, format='JPEG', quality=85)
        b64 = base64.standard_b64encode(buf.getvalue()).decode()
        frames.append({'frame_index':int(idx),'timestamp_sec':round(idx/fps,2),'b64':b64})
    cap.release()
    return frames, {'total_frames':total,'fps':fps,'duration_sec':round(total/fps,2)}

def compute_video_stats(video_path):
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    indices = np.linspace(0, total-1, min(30,total), dtype=int)
    bl,sl,ml,pg = [],[],[],None
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret: continue
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        bl.append(float(np.mean(gray))); sl.append(float(np.mean(hsv[:,:,1])))
        if pg is not None: ml.append(float(np.mean(cv2.absdiff(gray,pg))))
        pg = gray
    cap.release()
    ratio = round(w/h,3) if h else 0
    fmt   = 'vertical(9:16)' if ratio<0.7 else ('square(1:1)' if ratio<1.1 else 'horizontal(16:9)')
    return {'resolution':f'{w}x{h}','aspect_ratio':ratio,'duration_sec':round(total/fps,2),
            'fps':round(fps,2),'avg_brightness':round(np.mean(bl),1) if bl else 0,
            'avg_saturation':round(np.mean(sl),1) if sl else 0,
            'avg_motion_score':round(np.mean(ml),2) if ml else 0,'format_tag':fmt}

print('✅ 분석 엔진 로딩 완료!')

In [ ]:
# [Cell 4] Claude 멀티모달 분석 에이전트

SYSTEM_PROMPT = (
    '당신은 10년 경력의 디지털 마케팅 전문가이자 광고 효과 측정 전문가입니다.\\n'
    '마케팅 영상의 프레임을 분석하여 광고 효과를 정량적·정성적으로 평가합니다.\\n'
    '반드시 아래 JSON 형식으로만 응답하세요. 다른 텍스트는 절대 포함하지 마세요.\\n'
    '\\n'
    '{\\n'
    '  "overall_score": 0~100 사이의 정수 (종합 광고 효과 점수),\\n'
    '  "grade": "S/A/B/C/D 등급",\\n'
    '  "summary": "3줄 이내 핵심 총평",\\n'
    '  "scores": {\\n'
    '    "visual_impact":     {"score": 0~100, "comment": "평가 근거 1~2문장"},\\n'
    '    "message_clarity":   {"score": 0~100, "comment": "평가 근거 1~2문장"},\\n'
    '    "emotional_appeal":  {"score": 0~100, "comment": "평가 근거 1~2문장"},\\n'
    '    "cta_effectiveness": {"score": 0~100, "comment": "평가 근거 1~2문장"},\\n'
    '    "brand_consistency": {"score": 0~100, "comment": "평가 근거 1~2문장"},\\n'
    '    "production_quality":{"score": 0~100, "comment": "평가 근거 1~2문장"}\\n'
    '  },\\n'
    '  "platform_fit": {\\n'
    '    "youtube":   {"score": 0~100, "reason": "적합 이유"},\\n'
    '    "instagram": {"score": 0~100, "reason": "적합 이유"},\\n'
    '    "tiktok":    {"score": 0~100, "reason": "적합 이유"},\\n'
    '    "facebook":  {"score": 0~100, "reason": "적합 이유"}\\n'
    '  },\\n'
    '  "target_audience": {\\n'
    '    "primary": "추정 주요 타겟 (나이대, 관심사)",\\n'
    '    "resonance": "타겟 공감 가능성 평가"\\n'
    '  },\\n'
    '  "strengths":    ["강점1", "강점2", "강점3"],\\n'
    '  "weaknesses":   ["약점1", "약점2", "약점3"],\\n'
    '  "improvements": [\\n'
    '    {"priority": "HIGH/MED/LOW", "action": "개선 액션", "expected_impact": "기대 효과"}\\n'
    '  ],\\n'
    '  "roi_prediction": {\\n'
    '    "ctr_estimate": "예상 클릭률 범위 (예: 1.5~2.5%)",\\n'
    '    "conversion_potential": "LOW/MEDIUM/HIGH/VERY HIGH",\\n'
    '    "viral_potential":      "LOW/MEDIUM/HIGH/VERY HIGH",\\n'
    '    "rationale": "ROI 예측 근거 2~3문장"\\n'
    '  }\\n'
    '}\\n'
)

def build_user_message(frames, stats, meta, context):
    content = [{'type':'text','text':
        f'## 영상 기술 정보\n'
        f'- 해상도: {stats["resolution"]} / 포맷: {stats["format_tag"]}\n'
        f'- 재생 시간: {stats["duration_sec"]}초 / FPS: {stats["fps"]}\n'
        f'- 평균 밝기: {stats["avg_brightness"]} / 평균 채도: {stats["avg_saturation"]}\n'
        f'- 모션 강도: {stats["avg_motion_score"]} (높을수록 역동적)\n\n'
        f'## 마케터 컨텍스트\n{context}\n\n'
        f'아래 {len(frames)}개 프레임을 분석하세요:'}]
    for i,f in enumerate(frames):
        content.append({'type':'text','text':f'[프레임 {i+1} | {f["timestamp_sec"]}초]'})
        content.append({'type':'image','source':{'type':'base64','media_type':'image/jpeg','data':f['b64']}})
    content.append({'type':'text','text':'위 프레임들을 종합 분석하여 JSON으로만 응답하세요.'})
    return content

def analyze_with_claude(frames, stats, meta, context='(컨텍스트 없음)'):
    content  = build_user_message(frames, stats, meta, context)
    response = client.messages.create(
        model='claude-sonnet-4-20250514',
        max_tokens=4096,
        system=SYSTEM_PROMPT,
        messages=[{'role':'user','content':content}]
    )
    raw = response.content[0].text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    return json.loads(raw)

print('✅ 분석 에이전트 준비 완료!')


In [ ]:
# [Cell 5] 리포트 렌더러
from IPython.display import display, HTML

def score_bar(score, width=200):
    c = '#22c55e' if score>=80 else ('#f59e0b' if score>=60 else '#ef4444')
    return (f'<div style="display:inline-block;width:{width}px;height:10px;background:#e5e7eb;'
            f'border-radius:5px;vertical-align:middle;margin:0 8px">'
            f'<div style="width:{score}%;height:100%;background:{c};border-radius:5px"></div></div>'
            f'<b style="color:{c}">{score}</b>')

def grade_badge(g):
    cm = {'S':'#7c3aed','A':'#2563eb','B':'#16a34a','C':'#d97706','D':'#dc2626'}
    bg = cm.get(g[0],'#6b7280')
    return f'<span style="background:{bg};color:#fff;padding:4px 14px;border-radius:20px;font-size:1.3em;font-weight:bold">{g}</span>'

def priority_badge(p):
    cm = {'HIGH':'#dc2626','MED':'#d97706','LOW':'#16a34a'}
    return f'<span style="background:{cm.get(p,"#6b7280")};color:#fff;padding:2px 8px;border-radius:4px;font-size:.8em">{p}</span>'

def render_report(result, filename, stats):
    s=result.get('scores',{}); pf=result.get('platform_fit',{})
    roi=result.get('roi_prediction',{}); ta=result.get('target_audience',{}); imps=result.get('improvements',[])
    LM={'visual_impact':'🎯 시각적 임팩트','message_clarity':'💬 메시지 명확성',
        'emotional_appeal':'❤️ 감성 공명도','cta_effectiveness':'🔔 CTA 효과성',
        'brand_consistency':'🏷️ 브랜드 일관성','production_quality':'🎬 제작 완성도'}
    PI={'youtube':'▶️','instagram':'📸','tiktok':'🎵','facebook':'👥'}
    sr=''.join(f'<tr><td style="padding:8px 12px;font-weight:600">{LM.get(k,k)}</td>'
               f'<td style="padding:8px 12px">{score_bar(v["score"])}</td>'
               f'<td style="padding:8px 12px;color:#6b7280;font-size:.9em">{v["comment"]}</td></tr>' for k,v in s.items())
    pc=''.join(f'<div style="flex:1;min-width:130px;background:#f8fafc;border:1px solid #e2e8f0;border-radius:12px;padding:14px;text-align:center">'
               f'<div style="font-size:1.6em">{PI.get(k,"📱")}</div><div style="font-weight:700;margin:4px 0">{k.capitalize()}</div>'
               f'{score_bar(v["score"],120)}<br/><div style="font-size:.8em;color:#64748b;margin-top:6px">{v["reason"]}</div></div>' for k,v in pf.items())
    ir=''.join(f'<tr><td style="padding:8px 12px">{priority_badge(i["priority"])}</td>'
               f'<td style="padding:8px 12px;font-weight:500">{i["action"]}</td>'
               f'<td style="padding:8px 12px;color:#16a34a">{i["expected_impact"]}</td></tr>' for i in imps)
    sh=''.join(f'<div class="tag">{x}</div>' for x in result.get('strengths',[]))
    wh=''.join(f'<div class="tag weak">{x}</div>' for x in result.get('weaknesses',[]))
    html = f'''
<style>
  .report{{font-family:Segoe UI,sans-serif;max-width:920px;margin:0 auto;color:#1e293b}}
  .card{{background:#fff;border:1px solid #e2e8f0;border-radius:16px;padding:24px;margin-bottom:20px;box-shadow:0 2px 8px rgba(0,0,0,.06)}}
  table{{width:100%;border-collapse:collapse}} tr:nth-child(even){{background:#f8fafc}} th{{text-align:left}}
  .tag{{display:inline-block;background:#eff6ff;color:#1d4ed8;border-radius:6px;padding:3px 10px;margin:3px;font-size:.85em}}
  .weak{{background:#fff1f2!important;color:#be123c!important}}
</style>
<div class=report>
  <div class=card style=background:linear-gradient(135deg,#1e293b,#334155);color:#fff>
    <h2 style=margin:0>🎬 광고 효과 예측 리포트</h2>
    <div style=color:#94a3b8;font-size:.9em;margin-top:6px>📁 {filename} &nbsp;|&nbsp; ⏱ {stats["duration_sec"]}초 &nbsp;|&nbsp; 📐 {stats["resolution"]} ({stats["format_tag"]})</div>
    <div style=margin-top:20px;display:flex;align-items:center;gap:20px>
      <div style=text-align:center><div style=font-size:3em;font-weight:900;color:#facc15>{result["overall_score"]}</div><div style=color:#94a3b8>종합 점수 / 100</div></div>
      <div>{grade_badge(result["grade"])}</div>
      <div style=flex:1;color:#cbd5e1;font-size:.95em;line-height:1.7>{result["summary"]}</div>
    </div>
  </div>
  <div class=card><h3 style=margin:0 0 16px>📊 세부 평가 점수</h3><table>{sr}</table></div>
  <div class=card><h3 style=margin:0 0 16px>📱 플랫폼 적합성</h3><div style=display:flex;gap:12px;flex-wrap:wrap>{pc}</div></div>
  <div class=card style=display:flex;gap:24px;flex-wrap:wrap>
    <div style=flex:1;min-width:200px><h3 style=margin:0 0 12px;color:#16a34a>✅ 강점</h3>{sh}</div>
    <div style=flex:1;min-width:200px><h3 style=margin:0 0 12px;color:#dc2626>⚠️ 약점</h3>{wh}</div>
  </div>
  <div class=card><h3 style=margin:0 0 16px>🔧 개선 액션 플랜</h3>
    <table><tr style=background:#f1f5f9><th style=padding:8px 12px>우선순위</th><th style=padding:8px 12px>액션</th><th style=padding:8px 12px>기대 효과</th></tr>{ir}</table>
  </div>
  <div class=card style=background:#f0fdf4;border-color:#bbf7d0>
    <h3 style=margin:0 0 16px;color:#15803d>💰 ROI / 전환 예측</h3>
    <div style=display:flex;gap:24px;flex-wrap:wrap>
      <div><b>예상 CTR</b><br/><span style=font-size:1.4em;color:#15803d;font-weight:700>{roi.get("ctr_estimate","N/A")}</span></div>
      <div><b>전환 가능성</b><br/><span style=font-size:1.4em;color:#15803d;font-weight:700>{roi.get("conversion_potential","N/A")}</span></div>
      <div><b>바이럴 가능성</b><br/><span style=font-size:1.4em;color:#15803d;font-weight:700>{roi.get("viral_potential","N/A")}</span></div>
    </div>
    <p style=margin-top:14px;color:#166534;line-height:1.7>{roi.get("rationale","")}</p>
  </div>
  <div class=card><h3 style=margin:0 0 12px>🎯 추정 타겟 오디언스</h3>
    <p><b>주요 타겟:</b> {ta.get("primary","")}</p><p><b>공감 가능성:</b> {ta.get("resonance","")}</p>
  </div>
  <div style=text-align:center;color:#94a3b8;font-size:.8em;padding:12px>Powered by Claude claude-sonnet-4-20250514</div>
</div>'''
    display(HTML(html))

print('✅ 리포트 렌더러 준비 완료!')


In [ ]:
# [Cell 6] 영상 업로드 & 분석 실행 (메인)
import ipywidgets as widgets
from google.colab import files as colab_files
from IPython.display import display, HTML, clear_output

def run_analysis(video_path, filename, context, n_frames):
    print('\n🔍 영상 통계 계산 중...')
    stats = compute_video_stats(video_path)
    print(f'   해상도: {stats["resolution"]} | 길이: {stats["duration_sec"]}s | FPS: {stats["fps"]}')
    print(f'🎞️  프레임 추출 중 ({n_frames}개)...')
    frames, meta = extract_keyframes(video_path, n_frames)
    print(f'   {len(frames)}개 추출 완료')
    print('🤖 Claude AI 분석 중... (10~30초 소요)')
    t0     = time.time()
    result = analyze_with_claude(frames, stats, meta, context)
    print(f'   분석 완료! ({round(time.time()-t0,1)}초 소요)')
    render_report(result, filename, stats)
    out = f'/content/ad_analysis_{Path(filename).stem}.json'
    with open(out,'w',encoding='utf-8') as fp:
        json.dump({'filename':filename,'stats':stats,'analysis':result},fp,ensure_ascii=False,indent=2)
    print(f'\n💾 JSON 저장: {out}')
    return result

ctx_widget = widgets.Textarea(
    value='예시: 20~35세 여성 대상 뷰티 브랜드 신제품 런칭 광고. 핵심 메시지: 자연스러운 아름다움. 게재 채널: 인스타그램 릴스, 유튜브 쇼츠.',
    description='📝 광고 컨텍스트:',
    layout=widgets.Layout(width='100%',height='100px'),
    style={'description_width':'130px'}
)
frame_s = widgets.IntSlider(value=8,min=4,max=16,step=2,
    description='🎞️ 샘플 프레임:',
    style={'description_width':'130px'},layout=widgets.Layout(width='60%'))
btn     = widgets.Button(description='📁 영상 업로드 & 분석',button_style='primary',
    layout=widgets.Layout(width='240px',height='44px'))
out_w   = widgets.Output()

def on_click(b):
    with out_w:
        clear_output()
        print('📂 파일을 선택하세요 (mp4, mov, avi, mkv...)')
        uploaded = colab_files.upload()
        if not uploaded:
            print('❌ 파일이 업로드되지 않았습니다.'); return
        fname = list(uploaded.keys())[0]
        ctx   = ctx_widget.value.strip() or '(컨텍스트 없음)'
        try:
            run_analysis(f'/content/{fname}', fname, ctx, frame_s.value)
        except Exception as e:
            import traceback; print(f'❌ 오류: {e}'); traceback.print_exc()

btn.on_click(on_click)
display(widgets.VBox([
    widgets.HTML('<b>1단계:</b> 광고 컨텍스트 입력 (타겟, 채널, 핵심 메시지 등)'),
    ctx_widget,
    widgets.HTML('<br/><b>2단계:</b> 샘플 프레임 수 설정'),
    frame_s,
    widgets.HTML('<br/><b>3단계:</b> 영상 업로드 & 분석 실행'),
    btn, out_w
]))


In [ ]:
# [Cell 7] [선택] YouTube URL로 영상 분석
import subprocess, sys, glob
subprocess.check_call([sys.executable,'-m','pip','install','yt-dlp','-q'])

VIDEO_URL = 'https://www.youtube.com/watch?v=여기에_유튜브_ID_입력'
CONTEXT   = '25~35세 남성 대상 스포츠 브랜드 광고. 핵심 메시지: 한계를 넘어라.'
N_FRAMES  = 8

res = subprocess.run(
    ['yt-dlp','-f','mp4[height<=720]/best[height<=720]',
     '-o','/content/yt_video.%(ext)s','--no-playlist',VIDEO_URL],
    capture_output=True, text=True
)
if res.returncode != 0:
    print('❌ 다운로드 실패:', res.stderr[:500])
else:
    files = glob.glob('/content/yt_video.*')
    if files:
        vpath = files[0]
        print(f'✅ 다운로드 완료: {vpath}')
        run_analysis(vpath, Path(vpath).name, CONTEXT, N_FRAMES)
    else:
        print('❌ 파일을 찾을 수 없습니다.')


In [ ]:
# [Cell 8] [선택] 멀티 영상 A/B 배치 비교
from google.colab import files as colab_files
import pandas as pd

BATCH_CTX    = '동일 캠페인의 A/B 테스트용 광고 소재들. 타겟: 25~40세 직장인.'
BATCH_FRAMES = 6

print('📂 여러 영상을 한꺼번에 업로드하세요')
uploaded_batch = colab_files.upload()
batch_results  = []

for fname in uploaded_batch:
    vpath = f'/content/{fname}'
    print(f'\n🎬 분석 중: {fname}')
    try:
        st = compute_video_stats(vpath)
        fr, mt = extract_keyframes(vpath, BATCH_FRAMES)
        res = analyze_with_claude(fr, st, mt, BATCH_CTX)
        batch_results.append({'filename':fname,'stats':st,'result':res})
        print(f'   ✅ 점수: {res["overall_score"]} / 등급: {res["grade"]}')
    except Exception as e:
        print(f'   ❌ 오류: {e}')

if batch_results:
    rows = [{'파일명':br['filename'],'종합점수':br['result']['overall_score'],
             '등급':br['result']['grade'],
             '예상CTR':br['result'].get('roi_prediction',{}).get('ctr_estimate',''),
             '전환가능성':br['result'].get('roi_prediction',{}).get('conversion_potential',''),
             '바이럴가능성':br['result'].get('roi_prediction',{}).get('viral_potential','')}
            for br in sorted(batch_results,key=lambda x:x['result']['overall_score'],reverse=True)]
    display(HTML('<h3>📊 배치 비교 결과 (점수 높은 순)</h3>'))
    display(pd.DataFrame(rows).style.background_gradient(subset=['종합점수'],cmap='RdYlGn'))
    best = rows[0]
    br   = next(b for b in batch_results if b['filename']==best['파일명'])
    print(f'\n🏆 최고 점수: {br["filename"]} ({br["result"]["overall_score"]}점)')
    render_report(br['result'], br['filename'], br['stats'])


In [ ]:
# [Cell 1] 패키지 설치 (OpenAI 버전)
print('📦 필요한 패키지를 설치합니다...')

import subprocess
import sys

pkgs = [
    'openai',                    # 🔥 Anthropic → OpenAI
    'opencv-python-headless',
    'Pillow',
    'numpy',
    'ipywidgets',
    'tqdm',
    'yt-dlp'
]

for p in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', p, '-q'])

print('✅ 패키지 설치 완료!')

📦 필요한 패키지를 설치합니다...
✅ 패키지 설치 완료!


In [ ]:
# [Cell 2] OpenAI API 키 설정
import os
from getpass import getpass

print('🔑 OpenAI API 키를 입력하세요')
print('   발급: https://platform.openai.com/api-keys')

OPENAI_API_KEY = getpass('API Key: ')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# 🔍 간단한 형식 검증
if OPENAI_API_KEY.startswith('sk-'):
    print('✅ API 키가 설정되었습니다!')
else:
    print('⚠️  API 키 형식을 확인하세요 (보통 sk- 로 시작합니다)')

🔑 OpenAI API 키를 입력하세요
   발급: https://platform.openai.com/api-keys
API Key: ··········
✅ API 키가 설정되었습니다!


In [ ]:
# [Cell 3] 분석 엔진 (OpenAI 버전)

import cv2, base64, json, time, numpy as np, os
from PIL import Image
from io import BytesIO
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

# 🔥 Anthropic 제거 → OpenAI 클라이언트로 변경
from openai import OpenAI
client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])


# 🎬 핵심 프레임 추출
def extract_keyframes(video_path, n_frames=8):
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30

    indices = np.linspace(0, total-1, n_frames, dtype=int)
    frames  = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue

        img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        img.thumbnail((1280,720))

        buf = BytesIO()
        img.save(buf, format='JPEG', quality=85)

        b64 = base64.standard_b64encode(buf.getvalue()).decode()

        frames.append({
            'frame_index': int(idx),
            'timestamp_sec': round(idx/fps, 2),
            'b64': b64
        })

    cap.release()

    return frames, {
        'total_frames': total,
        'fps': fps,
        'duration_sec': round(total/fps, 2)
    }


# 📊 영상 통계 분석
def compute_video_stats(video_path):
    cap   = cv2.VideoCapture(video_path)

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    indices = np.linspace(0, total-1, min(30, total), dtype=int)

    bl, sl, ml = [], [], []
    pg = None

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        bl.append(float(np.mean(gray)))
        sl.append(float(np.mean(hsv[:,:,1])))

        if pg is not None:
            ml.append(float(np.mean(cv2.absdiff(gray, pg))))

        pg = gray

    cap.release()

    ratio = round(w/h, 3) if h else 0

    fmt = (
        'vertical(9:16)' if ratio < 0.7 else
        ('square(1:1)' if ratio < 1.1 else 'horizontal(16:9)')
    )

    return {
        'resolution': f'{w}x{h}',
        'aspect_ratio': ratio,
        'duration_sec': round(total/fps, 2),
        'fps': round(fps, 2),
        'avg_brightness': round(np.mean(bl), 1) if bl else 0,
        'avg_saturation': round(np.mean(sl), 1) if sl else 0,
        'avg_motion_score': round(np.mean(ml), 2) if ml else 0,
        'format_tag': fmt
    }

print('✅ 분석 엔진 로딩 완료! (OpenAI 버전)')

✅ 분석 엔진 로딩 완료! (OpenAI 버전)


In [ ]:
# [Cell 4] OpenAI 멀티모달 분석 에이전트

import json

SYSTEM_PROMPT = (
    '당신은 10년 경력의 디지털 마케팅 전문가이자 광고 효과 측정 전문가입니다.\n'
    '마케팅 영상의 프레임을 분석하여 광고 효과를 정량적·정성적으로 평가합니다.\n'
    '반드시 아래 JSON 형식으로만 응답하세요. 다른 텍스트는 절대 포함하지 마세요.\n'
    '\n'
    '{\n'
    '  "overall_score": 0~100 사이의 정수,\n'
    '  "grade": "S/A/B/C/D 등급",\n'
    '  "summary": "3줄 이내 핵심 총평",\n'
    '  "scores": {\n'
    '    "visual_impact":     {"score": 0~100, "comment": ""},\n'
    '    "message_clarity":   {"score": 0~100, "comment": ""},\n'
    '    "emotional_appeal":  {"score": 0~100, "comment": ""},\n'
    '    "cta_effectiveness": {"score": 0~100, "comment": ""},\n'
    '    "brand_consistency": {"score": 0~100, "comment": ""},\n'
    '    "production_quality":{"score": 0~100, "comment": ""}\n'
    '  },\n'
    '  "platform_fit": {\n'
    '    "youtube":   {"score": 0~100, "reason": ""},\n'
    '    "instagram": {"score": 0~100, "reason": ""},\n'
    '    "tiktok":    {"score": 0~100, "reason": ""},\n'
    '    "facebook":  {"score": 0~100, "reason": ""}\n'
    '  },\n'
    '  "target_audience": {\n'
    '    "primary": "",\n'
    '    "resonance": ""\n'
    '  },\n'
    '  "strengths": [],\n'
    '  "weaknesses": [],\n'
    '  "improvements": [\n'
    '    {"priority": "HIGH/MED/LOW", "action": "", "expected_impact": ""}\n'
    '  ],\n'
    '  "roi_prediction": {\n'
    '    "ctr_estimate": "",\n'
    '    "conversion_potential": "LOW/MEDIUM/HIGH/VERY HIGH",\n'
    '    "viral_potential": "LOW/MEDIUM/HIGH/VERY HIGH",\n'
    '    "rationale": ""\n'
    '  }\n'
    '}\n'
)


# 🔥 OpenAI용 메시지 구성
def build_user_message(frames, stats, meta, context):
    content = []

    # 텍스트
    content.append({
        "type": "input_text",
        "text":
        f"## 영상 기술 정보\n"
        f"- 해상도: {stats['resolution']} / 포맷: {stats['format_tag']}\n"
        f"- 재생 시간: {stats['duration_sec']}초 / FPS: {stats['fps']}\n"
        f"- 평균 밝기: {stats['avg_brightness']} / 평균 채도: {stats['avg_saturation']}\n"
        f"- 모션 강도: {stats['avg_motion_score']}\n\n"
        f"## 마케터 컨텍스트\n{context}\n\n"
        f"아래 {len(frames)}개 프레임을 분석하세요:"
    })

    # 이미지
    for i, f in enumerate(frames):
        content.append({
            "type": "input_text",
            "text": f"[프레임 {i+1} | {f['timestamp_sec']}초]"
        })

        content.append({
            "type": "input_image",
            "image_base64": f['b64']
        })

    content.append({
        "type": "input_text",
        "text": "위 프레임들을 종합 분석하여 JSON으로만 응답하세요."
    })

    return content


# 🔥 핵심: OpenAI 분석 함수
def analyze_with_openai(frames, stats, meta, context='(컨텍스트 없음)'):
    content = build_user_message(frames, stats, meta, context)

    response = client.responses.create(
        model="gpt-4.1",
        input=[{
            "role": "user",
            "content": content
        }],
        max_output_tokens=4000,

        # 🔥 JSON 강제 출력 (매우 중요)
        response_format={"type": "json_object"}
    )

    raw = response.output[0].content[0].text.strip()

    return json.loads(raw)


print('✅ OpenAI 분석 에이전트 준비 완료!')

✅ OpenAI 분석 에이전트 준비 완료!


In [ ]:
# [Cell 5] 리포트 렌더러 (OpenAI 버전)

from IPython.display import display, HTML

def score_bar(score, width=200):
    c = '#22c55e' if score>=80 else ('#f59e0b' if score>=60 else '#ef4444')
    return (f'<div style="display:inline-block;width:{width}px;height:10px;background:#e5e7eb;'
            f'border-radius:5px;vertical-align:middle;margin:0 8px">'
            f'<div style="width:{score}%;height:100%;background:{c};border-radius:5px"></div></div>'
            f'<b style="color:{c}">{score}</b>')

def grade_badge(g):
    cm = {'S':'#7c3aed','A':'#2563eb','B':'#16a34a','C':'#d97706','D':'#dc2626'}
    bg = cm.get(g[0],'#6b7280')
    return f'<span style="background:{bg};color:#fff;padding:4px 14px;border-radius:20px;font-size:1.3em;font-weight:bold">{g}</span>'

def priority_badge(p):
    cm = {'HIGH':'#dc2626','MED':'#d97706','LOW':'#16a34a'}
    return f'<span style="background:{cm.get(p,"#6b7280")};color:#fff;padding:2px 8px;border-radius:4px;font-size:.8em">{p}</span>'

def render_report(result, filename, stats):
    s=result.get('scores',{})
    pf=result.get('platform_fit',{})
    roi=result.get('roi_prediction',{})
    ta=result.get('target_audience',{})
    imps=result.get('improvements',[])

    LM={'visual_impact':'🎯 시각적 임팩트','message_clarity':'💬 메시지 명확성',
        'emotional_appeal':'❤️ 감성 공명도','cta_effectiveness':'🔔 CTA 효과성',
        'brand_consistency':'🏷️ 브랜드 일관성','production_quality':'🎬 제작 완성도'}

    PI={'youtube':'▶️','instagram':'📸','tiktok':'🎵','facebook':'👥'}

    sr=''.join(
        f'<tr><td style="padding:8px 12px;font-weight:600">{LM.get(k,k)}</td>'
        f'<td style="padding:8px 12px">{score_bar(v["score"])}</td>'
        f'<td style="padding:8px 12px;color:#6b7280;font-size:.9em">{v["comment"]}</td></tr>'
        for k,v in s.items()
    )

    pc=''.join(
        f'<div style="flex:1;min-width:130px;background:#f8fafc;border:1px solid #e2e8f0;border-radius:12px;padding:14px;text-align:center">'
        f'<div style="font-size:1.6em">{PI.get(k,"📱")}</div>'
        f'<div style="font-weight:700;margin:4px 0">{k.capitalize()}</div>'
        f'{score_bar(v["score"],120)}<br/>'
        f'<div style="font-size:.8em;color:#64748b;margin-top:6px">{v["reason"]}</div></div>'
        for k,v in pf.items()
    )

    ir=''.join(
        f'<tr><td style="padding:8px 12px">{priority_badge(i["priority"])}</td>'
        f'<td style="padding:8px 12px;font-weight:500">{i["action"]}</td>'
        f'<td style="padding:8px 12px;color:#16a34a">{i["expected_impact"]}</td></tr>'
        for i in imps
    )

    sh=''.join(f'<div class="tag">{x}</div>' for x in result.get('strengths',[]))
    wh=''.join(f'<div class="tag weak">{x}</div>' for x in result.get('weaknesses',[]))

    html = f'''
<style>
  .report{{font-family:Segoe UI,sans-serif;max-width:920px;margin:0 auto;color:#1e293b}}
  .card{{background:#fff;border:1px solid #e2e8f0;border-radius:16px;padding:24px;margin-bottom:20px;box-shadow:0 2px 8px rgba(0,0,0,.06)}}
  table{{width:100%;border-collapse:collapse}} tr:nth-child(even){{background:#f8fafc}} th{{text-align:left}}
  .tag{{display:inline-block;background:#eff6ff;color:#1d4ed8;border-radius:6px;padding:3px 10px;margin:3px;font-size:.85em}}
  .weak{{background:#fff1f2!important;color:#be123c!important}}
</style>

<div class=report>

  <div class=card style=background:linear-gradient(135deg,#1e293b,#334155);color:#fff>
    <h2 style=margin:0>🎬 광고 효과 예측 리포트</h2>
    <div style=color:#94a3b8;font-size:.9em;margin-top:6px>
      📁 {filename} | ⏱ {stats["duration_sec"]}초 | 📐 {stats["resolution"]} ({stats["format_tag"]})
    </div>

    <div style=margin-top:20px;display:flex;align-items:center;gap:20px>
      <div style=text-align:center>
        <div style=font-size:3em;font-weight:900;color:#facc15>{result["overall_score"]}</div>
        <div style=color:#94a3b8>종합 점수 / 100</div>
      </div>

      <div>{grade_badge(result["grade"])}</div>

      <div style=flex:1;color:#cbd5e1;font-size:.95em;line-height:1.7>
        {result["summary"]}
      </div>
    </div>
  </div>

  <div class=card>
    <h3>📊 세부 평가 점수</h3>
    <table>{sr}</table>
  </div>

  <div class=card>
    <h3>📱 플랫폼 적합성</h3>
    <div style="display:flex;gap:12px;flex-wrap:wrap">{pc}</div>
  </div>

  <div class=card style=display:flex;gap:24px;flex-wrap:wrap>
    <div style=flex:1><h3 style=color:#16a34a>✅ 강점</h3>{sh}</div>
    <div style=flex:1><h3 style=color:#dc2626>⚠️ 약점</h3>{wh}</div>
  </div>

  <div class=card>
    <h3>🔧 개선 액션 플랜</h3>
    <table>{ir}</table>
  </div>

  <div class=card style=background:#f0fdf4>
    <h3>💰 ROI / 전환 예측</h3>
    <p>{roi.get("rationale","")}</p>
  </div>

  <div class=card>
    <h3>🎯 타겟 오디언스</h3>
    <p>{ta.get("primary","")}</p>
  </div>

  <div style=text-align:center;color:#94a3b8;font-size:.8em;padding:12px>
    Powered by OpenAI GPT-4.1
  </div>

</div>
'''

    display(HTML(html))


print('✅ 리포트 렌더러 준비 완료! (OpenAI)')

✅ 리포트 렌더러 준비 완료! (OpenAI)


In [ ]:
# [Cell 6] 영상 업로드 & 분석 실행 (메인 - OpenAI 버전)

import ipywidgets as widgets
from google.colab import files as colab_files
from IPython.display import display, HTML, clear_output

def run_analysis(video_path, filename, context, n_frames):
    print('\n🔍 영상 통계 계산 중...')
    stats = compute_video_stats(video_path)

    print(f'   해상도: {stats["resolution"]} | 길이: {stats["duration_sec"]}s | FPS: {stats["fps"]}')

    print(f'🎞️  프레임 추출 중 ({n_frames}개)...')
    frames, meta = extract_keyframes(video_path, n_frames)
    print(f'   {len(frames)}개 추출 완료')

    # 🔥 변경 1: 메시지 수정
    print('🤖 OpenAI GPT 분석 중... (10~30초 소요)')

    t0 = time.time()

    # 🔥 변경 2: 함수 교체 (핵심)
    result = analyze_with_openai(frames, stats, meta, context)

    print(f'   분석 완료! ({round(time.time()-t0,1)}초 소요)')

    render_report(result, filename, stats)

    # JSON 저장
    out = f'/content/ad_analysis_{Path(filename).stem}.json'
    with open(out, 'w', encoding='utf-8') as fp:
        json.dump({
            'filename': filename,
            'stats': stats,
            'analysis': result
        }, fp, ensure_ascii=False, indent=2)

    print(f'\n💾 JSON 저장: {out}')

    return result


# UI 구성
ctx_widget = widgets.Textarea(
    value='예시: 20~35세 여성 대상 뷰티 브랜드 신제품 런칭 광고. 핵심 메시지: 자연스러운 아름다움. 게재 채널: 인스타그램 릴스, 유튜브 쇼츠.',
    description='📝 광고 컨텍스트:',
    layout=widgets.Layout(width='100%', height='100px'),
    style={'description_width': '130px'}
)

frame_s = widgets.IntSlider(
    value=8,
    min=4,
    max=16,
    step=2,
    description='🎞️ 샘플 프레임:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='60%')
)

btn = widgets.Button(
    description='📁 영상 업로드 & 분석',
    button_style='primary',
    layout=widgets.Layout(width='240px', height='44px')
)

out_w = widgets.Output()


def on_click(b):
    with out_w:
        clear_output()

        print('📂 파일을 선택하세요 (mp4, mov, avi, mkv...)')

        uploaded = colab_files.upload()

        if not uploaded:
            print('❌ 파일이 업로드되지 않았습니다.')
            return

        fname = list(uploaded.keys())[0]
        ctx = ctx_widget.value.strip() or '(컨텍스트 없음)'

        try:
            run_analysis(f'/content/{fname}', fname, ctx, frame_s.value)
        except Exception as e:
            import traceback
            print(f'❌ 오류: {e}')
            traceback.print_exc()


btn.on_click(on_click)

display(widgets.VBox([
    widgets.HTML('<b>1단계:</b> 광고 컨텍스트 입력 (타겟, 채널, 핵심 메시지 등)'),
    ctx_widget,
    widgets.HTML('<br/><b>2단계:</b> 샘플 프레임 수 설정'),
    frame_s,
    widgets.HTML('<br/><b>3단계:</b> 영상 업로드 & 분석 실행'),
    btn,
    out_w
]))

In [ ]:
# [Cell 7] [선택] YouTube URL로 영상 분석 (안정화 버전)

import subprocess
import sys
import glob
from pathlib import Path

# 🔥 yt-dlp 설치 (이미 설치돼 있어도 안전)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'yt-dlp', '-q'])

# ===== 사용자 입력 =====
VIDEO_URL = 'https://www.youtube.com/watch?v=여기에_유튜브_ID_입력'
CONTEXT   = '25~35세 남성 대상 스포츠 브랜드 광고. 핵심 메시지: 한계를 넘어라.'
N_FRAMES  = 8
# ======================

print('📥 YouTube 영상 다운로드 중...')

cmd = [
    'yt-dlp',
    '-f', 'mp4[height<=720]/best[height<=720]',
    '-o', '/content/yt_video.%(ext)s',
    '--no-playlist',
    VIDEO_URL
]

res = subprocess.run(cmd, capture_output=True, text=True)

if res.returncode != 0:
    print('❌ 다운로드 실패')
    print(res.stderr[:500])
else:
    print('✅ 다운로드 성공')

    # 🔍 파일 찾기 (확장자 대응)
    files = glob.glob('/content/yt_video.*')

    if not files:
        print('❌ 다운로드된 파일을 찾을 수 없습니다.')
    else:
        vpath = files[0]
        print(f'🎬 분석 시작: {vpath}')

        try:
            # 🔥 여기 중요: 이미 OpenAI로 바뀐 run_analysis 사용
            run_analysis(vpath, Path(vpath).name, CONTEXT, N_FRAMES)

        except Exception as e:
            import traceback
            print('❌ 분석 중 오류 발생:', e)
            traceback.print_exc()

📥 YouTube 영상 다운로드 중...
✅ 다운로드 성공
🎬 분석 시작: /content/yt_video.mp4

🔍 영상 통계 계산 중...
   해상도: 360x640 | 길이: 90.5s | FPS: 30.0
🎞️  프레임 추출 중 (8개)...
   8개 추출 완료
🤖 OpenAI GPT 분석 중... (10~30초 소요)
❌ 분석 중 오류 발생: Responses.create() got an unexpected keyword argument 'response_format'


Traceback (most recent call last):
  File "/tmp/ipykernel_6550/4189356055.py", line 46, in <cell line: 0>
    run_analysis(vpath, Path(vpath).name, CONTEXT, N_FRAMES)
  File "/tmp/ipykernel_6550/1817028689.py", line 23, in run_analysis
    result = analyze_with_openai(frames, stats, meta, context)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_6550/994728326.py", line 88, in analyze_with_openai
    response = client.responses.create(
               ^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: Responses.create() got an unexpected keyword argument 'response_format'


In [ ]:
# [Cell 8] [선택] 멀티 영상 A/B 배치 비교 (OpenAI 버전)

from google.colab import files as colab_files
import pandas as pd

BATCH_CTX    = '동일 캠페인의 A/B 테스트용 광고 소재들. 타겟: 25~40세 직장인.'
BATCH_FRAMES = 6

print('📂 여러 영상을 한꺼번에 업로드하세요')

uploaded_batch = colab_files.upload()
batch_results  = []

for fname in uploaded_batch:
    vpath = f'/content/{fname}'
    print(f'\n🎬 분석 중: {fname}')

    try:
        st = compute_video_stats(vpath)
        fr, mt = extract_keyframes(vpath, BATCH_FRAMES)

        # 🔥 핵심 변경 (딱 이 한 줄)
        res = analyze_with_openai(fr, st, mt, BATCH_CTX)

        batch_results.append({
            'filename': fname,
            'stats': st,
            'result': res
        })

        print(f'   ✅ 점수: {res["overall_score"]} / 등급: {res["grade"]}')

    except Exception as e:
        print(f'   ❌ 오류: {e}')


if batch_results:
    rows = [
        {
            '파일명': br['filename'],
            '종합점수': br['result']['overall_score'],
            '등급': br['result']['grade'],
            '예상CTR': br['result'].get('roi_prediction', {}).get('ctr_estimate', ''),
            '전환가능성': br['result'].get('roi_prediction', {}).get('conversion_potential', ''),
            '바이럴가능성': br['result'].get('roi_prediction', {}).get('viral_potential', '')
        }
        for br in sorted(batch_results, key=lambda x: x['result']['overall_score'], reverse=True)
    ]

    display(HTML('<h3>📊 배치 비교 결과 (점수 높은 순)</h3>'))

    display(
        pd.DataFrame(rows)
        .style
        .background_gradient(subset=['종합점수'], cmap='RdYlGn')
    )

    best = rows[0]
    br   = next(b for b in batch_results if b['filename'] == best['파일명'])

    print(f'\n🏆 최고 점수: {br["filename"]} ({br["result"]["overall_score"]}점)')

    render_report(br['result'], br['filename'], br['stats'])

📂 여러 영상을 한꺼번에 업로드하세요


KeyboardInterrupt: 